# FTransform

> Reusable, reversible data transformations and pipelines, with native Julia multiple dispatch in place of plum/metaclass tricks — a Julia port of AnswerDotAI/fasttransform.

In [ ]:
#| hide
using FTransform

## Install

```julia
using Pkg
Pkg.add(url="https://github.com/MichaelAnokye56/FTransform.jl")
```

(Not yet registered in the Julia General registry.)

## How to use

Wrap a plain function:

In [ ]:
#| hide
add_one(x::Int) = x + 1
t1 = transform(add_one)
@assert t1(2) == 3
@assert t1(2.0) == 2.0  # no matching method for Float64 -> passthrough

Reversible:

In [ ]:
#| hide
t2 = transform(x -> x * 2; dec = x -> x ÷ 2)
@assert t2(2) == 4
@assert decode(t2, t2(2)) == 2

Type-based dispatch — just a named function with multiple methods; this is
exactly what fasttransform's own `plum` dependency exists to fake, and
Julia's native multiple dispatch already does it for free:

In [ ]:
#| hide
incdisp(x::Int) = x + 1
incdisp(x::AbstractString) = x * "a"
t3 = transform(incdisp)
@assert t3(5) == 6
@assert t3("b") == "ba"

Defining your own `encodes`/`decodes`/`setups!` for a new `Transform`
subtype needs `import`, not just `using` — see the README's "A Julia
gotcha" for why this matters:

In [ ]:
#| hide
import FTransform: encodes, decodes, setups!

mutable struct NormalizeMean <: Transform
    mean::Float64
    std::Float64
    NormalizeMean() = new(0.0, 1.0)
end
setups!(t::NormalizeMean, items) = (t.mean = sum(items) / length(items))
encodes(t::NormalizeMean, x) = (x - t.mean) / t.std
decodes(t::NormalizeMean, x) = x * t.std + t.mean
normalize = setup!(NormalizeMean(), [1, 2, 3, 4, 5])
@assert normalize.mean == 3.0

`Pipeline` composes several transforms; `decode` undoes them in reverse order:

In [ ]:
#| hide
p = Pipeline([t2, normalize])
v = p(5)
@assert decode(p, v) == 5.0